In [ ]:
import cv2
import os
import numpy as np
from pdf2image import convert_from_path
from PIL import Image

from paddleocr import PaddleOCR
import pytesseract
import easyocr

os.environ["TOKENIZERS_PARALLELISM"] = "false"

### 전처리

In [ ]:
# 이미지 전처리 함수
def preprocess_image(img):
    img_array = np.array(img)
    
    # 그레이스케일 변환
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array
    
    # 이진화
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    
    # 노이즈 제거
    denoised = cv2.fastNlMeansDenoising(binary, None, 10, 7, 21)
    
    # 이미지 선명하게
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(denoised, -1, kernel)
    
    # NumPy 배열을 PIL 이미지로 변환
    processed_img = Image.fromarray(sharpened)
    
    return processed_img

### 모델

In [ ]:
# PDF 경로
# pdf_path = "../../sample/term_sheet/sample_termsheet.pdf"
pdf_path = "./termsheet/1000000019362_NH_거래확인서_20231215.pdf"
# PDF를 이미지로 변환
images = convert_from_path(pdf_path, dpi=300)

# 각 이미지 전처리
processed_images = []
for img in images:
    processed = preprocess_image(img)
    processed_images.append(processed)

print(f"총 {len(processed_images)}개 페이지가 처리되었습니다.")

#### 1. paddle OCR

2시간 >> 20분 (GPU: true)

In [ ]:
ocr = PaddleOCR(
    ocr_version='PP-OCRv4',
    lang='korean',
    use_gpu=True,
    
    use_mp=True,            # 멀티 프로세싱 사용
    total_process_num=4,    # 프로세스 수
    rec_batch_num=8,        # 배치 처리 크기
    
    cls=True,               # 회전된 텍스트 감지
    det_db_thresh=0.25,     # 기본값: 0.3, 낮출수록 더 많은 텍스트 감지 (0.2-0.4)
    det_db_box_thresh=0.55, # 기본값: 0.6, 낮출수록 더 많은 박스 감지 (0.5-0.7)
    det_db_unclip_ratio=1.8 # 기본값: 1.5, 높일수록 박스 크기 증가 (1.5-2.0)
)

In [ ]:
# 출력 폴더 생성
output_folder = "./paddle_output"
os.makedirs(output_folder, exist_ok=True)

total_tokens = 0
all_texts = []

# 각 페이지에 대해 OCR 수행
for i, img in enumerate(images):
    print(f"페이지 {i+1} 처리 중...")
    
    # 이미지를 numpy 배열로 변환
    img_array = np.array(img)
    
    # OCR 실행
    result = ocr.ocr(img_array, cls=True)
    
    # 기본 텍스트 추출
    page_text = ""
    if result[0]:
        for line in result[0]:
            if line:
                page_text += line[1][0] + " "
    
    all_texts.append(page_text)
    
    # 결과 출력
    print(f"페이지 {i+1} 결과:")
    print(page_text)

    # 페이지별 단어 수 계산
    words = page_text.split()
    page_tokens = len(words)
    total_tokens += page_tokens
    print(f"감지된 단어 수: {page_tokens}")
    print("-" * 50)

# 전체 토큰 수 출력
print(f"총 인식된 단어 수: {total_tokens}")

# 파일 이름 생성
pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
ocr_engine_name = "paddle"
txt_file_path = os.path.join(output_folder, f"{pdf_filename}_{ocr_engine_name}_ocr.txt")

# OCR 결과를 텍스트 파일로 저장
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))

print(f"OCR 결과가 {txt_file_path}에 저장되었습니다.")

#### 2. Tesseract OCR

##### Page Segmentation modes

--psm 0: 방향 및 스크립트 감지만  
기능: 텍스트를 인식하지 않고 페이지 방향과 언어 종류만 감지  

--psm 3: 자동 페이지 분할 (기본값)  
기능: 자동으로 페이지 레이아웃을 분석하여 텍스트 영역 구분  

--psm 4: 단일 컬럼 텍스트로 처리  
기능: 페이지 전체를 하나의 수직 컬럼으로 간주  

--psm 6: 단일 텍스트 블록으로 처리  
기능: 전체 이미지를 하나의 통일된 텍스트 블록으로 처리  

--psm 11: 구조 없는 텍스트로 처리 (표)  
기능: 특별한 구조를 가정하지 않고 텍스트 그대로 인식  

--psm 12: 희소 텍스트로 처리  
기능: 페이지에 텍스트가 드물게 존재하는 경우에 최적화  

##### OCR Engine modes

--oem 0: 레거시 엔진만  

--oem 1: 신경망 LSTM 엔진만  

--oem 2: 레거시+LSTM 함께 사용  

--oem 3: 자동 선택 (기본값)  

2분 (GPU 지원 X)

In [ ]:
# OCR 설정
OCR_CONFIG = '--psm 3 --oem 3'
OCR_LANG = 'eng+kor'

def detect_tables(image):
    """이미지에서 표 영역 감지"""
    # 이미지 전처리
    img_array = np.array(image)
    
    # 이미지가 이미 그레이스케일인지 확인
    if len(img_array.shape) == 3 and img_array.shape[2] == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_array
    
    threshold = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)[1]
    
    # 수평/수직 라인 감지
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    
    horizontal_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    vertical_lines = cv2.morphologyEx(threshold, cv2.MORPH_OPEN, vertical_kernel, iterations=1)
    
    # 표 테두리 감지
    table_boundaries = cv2.addWeighted(horizontal_lines, 0.5, vertical_lines, 0.5, 0.0)
    
    # 테두리에서 표 영역 찾기
    contours, _ = cv2.findContours(table_boundaries, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    table_regions = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w > 100 and h > 100:  # 작은 영역 필터링
            table_regions.append((x, y, x+w, y+h))
    
    return table_regions

def ocr_with_table_detection(image):
    """표 감지와 행열 순서 정렬을 적용한 OCR 처리"""
    img_array = np.array(image)
    table_regions = detect_tables(img_array)
    
    if not table_regions:
        # 표가 없으면 전체 이미지 OCR
        return pytesseract.image_to_string(image, lang=OCR_LANG, config=OCR_CONFIG)
    
    # 표가 있으면 영역별 처리 후 Y좌표 순으로 조합
    text_blocks = []
    table_regions = sorted(table_regions, key=lambda x: x[1])
    
    # 전체 영역을 순회하며 표/텍스트 구분 처리
    height = img_array.shape[0]
    current_y = 0
    
    for x1, y1, x2, y2 in table_regions:
        # 표 이전 일반 텍스트
        if y1 > current_y + 10:
            text_img = img_array[current_y:y1, :]
            text = pytesseract.image_to_string(Image.fromarray(text_img), lang=OCR_LANG, config=OCR_CONFIG)
            if text.strip():
                text_blocks.append(text.strip())
        
        # 표 영역 (행열 정렬)
        table_img = img_array[y1:y2, x1:x2]
        table_text = _process_table_with_row_order(table_img)
        if table_text.strip():
            text_blocks.append(f"--- 표 내용 ---\n{table_text}\n--- 표 끝 ---")
        
        current_y = y2
    
    # 마지막 표 이후 텍스트
    if current_y < height - 10:
        text_img = img_array[current_y:, :]
        text = pytesseract.image_to_string(Image.fromarray(text_img), lang=OCR_LANG, config=OCR_CONFIG)
        if text.strip():
            text_blocks.append(text.strip())
    
    return '\n\n'.join(text_blocks)

def _process_table_with_row_order(table_img):
    """표 영역 행열 순서 정렬"""
    data = pytesseract.image_to_data(
        Image.fromarray(table_img), 
        lang=OCR_LANG, config=OCR_CONFIG,
        output_type=pytesseract.Output.DICT
    )
    
    # 신뢰도 30 이상 텍스트만 수집
    boxes = [(data['text'][i].strip(), data['left'][i], data['top'][i])
             for i in range(len(data['text']))
             if int(data['conf'][i]) > 30 and data['text'][i].strip()]
    
    if not boxes:
        return pytesseract.image_to_string(Image.fromarray(table_img), lang=OCR_LANG, config=OCR_CONFIG)
    
    # Y좌표로 행 그룹핑 후 정렬
    rows = {}
    for text, x, y in boxes:
        row_key = y // 20 * 20  # 20픽셀 단위로 그룹핑
        if row_key not in rows:
            rows[row_key] = []
        rows[row_key].append((text, x))
    
    # 행별로 X좌표 정렬하여 텍스트 조합
    result = []
    for row_y in sorted(rows.keys()):
        row_texts = [text for text, x in sorted(rows[row_y], key=lambda item: item[1])]
        result.append(' '.join(row_texts))
    
    return '\n'.join(result)

# 메인 실행부
output_folder = "./tesseract_output"
os.makedirs(output_folder, exist_ok=True)

all_texts = []
total_tokens = 0

for i, img in enumerate(processed_images):
    print(f"페이지 {i+1} 처리 중...")
    
    text = ocr_with_table_detection(img)
    all_texts.append(text)
    
    # 토큰 수 계산
    word_count = len(text.split())
    estimated_tokens = int(word_count * 1.3)
    total_tokens += estimated_tokens
    
    print(f"단어 수: {word_count}, 예상 토큰: {estimated_tokens}")
    print("-" * 50)

# 결과 저장
pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
txt_file_path = os.path.join(output_folder, f"{pdf_filename}_tesseract_ocr.txt")

with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))

print(f"총 단어: {sum(len(text.split()) for text in all_texts)}")
print(f"총 토큰: {total_tokens}")
print(f"결과 저장: {txt_file_path}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_text_detection(image, save_path=None):
    """텍스트 감지 박스를 시각화"""
    img_array = np.array(image)
    
    # Tesseract로 텍스트 박스 감지
    data = pytesseract.image_to_data(
        image, 
        lang=OCR_LANG, 
        config=OCR_CONFIG,
        output_type=pytesseract.Output.DICT
    )
    
    # 시각화 준비
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # 원본 이미지
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title('원본 이미지')
    ax1.axis('off')
    
    # 텍스트 박스가 있는 이미지
    ax2.imshow(img_array, cmap='gray')
    ax2.set_title('텍스트 감지 박스')
    ax2.axis('off')
    
    # 신뢰도별 색상 설정
    def get_color_by_confidence(conf):
        if conf >= 80:
            return 'green'      # 높은 신뢰도
        elif conf >= 50:
            return 'orange'     # 중간 신뢰도
        else:
            return 'red'        # 낮은 신뢰도
    
    # 텍스트 박스 그리기
    detected_texts = []
    for i in range(len(data['text'])):
        conf = int(data['conf'][i])
        text = data['text'][i].strip()
        
        if conf > 10 and text:  # 신뢰도 10 이상인 텍스트만
            x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
            
            # 박스 그리기
            color = get_color_by_confidence(conf)
            rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor=color, facecolor='none')
            ax2.add_patch(rect)
            
            # 신뢰도 텍스트 추가
            ax2.text(x, y-5, f'{conf}%', fontsize=8, color=color, weight='bold')
            
            detected_texts.append(f"'{text}' (신뢰도: {conf}%)")
    
    # 범례 추가
    green_patch = patches.Patch(color='green', label='높은 신뢰도 (80%+)')
    orange_patch = patches.Patch(color='orange', label='중간 신뢰도 (50-79%)')
    red_patch = patches.Patch(color='red', label='낮은 신뢰도 (10-49%)')
    ax2.legend(handles=[green_patch, orange_patch, red_patch], loc='upper right')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"시각화 이미지 저장: {save_path}")
    
    plt.show()
    
    return detected_texts

def visualize_table_detection(image, save_path=None):
    """표 감지 결과 시각화"""
    img_array = np.array(image)
    table_regions = detect_tables(img_array)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # 원본 이미지
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title('원본 이미지')
    ax1.axis('off')
    
    # 표 감지 결과
    ax2.imshow(img_array, cmap='gray')
    ax2.set_title(f'표 감지 결과 ({len(table_regions)}개 표 발견)')
    ax2.axis('off')
    
    # 표 영역 박스 그리기
    for i, (x1, y1, x2, y2) in enumerate(table_regions):
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                               linewidth=3, edgecolor='blue', facecolor='none')
        ax2.add_patch(rect)
        
        # 표 번호 표시
        ax2.text(x1, y1-10, f'표 {i+1}', fontsize=12, color='blue', weight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"표 감지 시각화 저장: {save_path}")
    
    plt.show()
    
    return table_regions

# 시각화 테스트 함수
def test_text_detection_visualization():
    """첫 번째 페이지로 텍스트 감지 테스트"""
    if processed_images:
        print("첫 번째 페이지의 텍스트 감지 결과를 시각화합니다...")
        
        # 출력 폴더 생성
        viz_folder = "./visualization_output"
        os.makedirs(viz_folder, exist_ok=True)
        
        # 텍스트 감지 시각화
        text_save_path = os.path.join(viz_folder, "text_detection_page1.png")
        detected_texts = visualize_text_detection(processed_images[0], text_save_path)
        
        print("\n" + "="*50)
        
        # 표 감지 시각화
        table_save_path = os.path.join(viz_folder, "table_detection_page1.png")
        table_regions = visualize_table_detection(processed_images[0], table_save_path)
        
        print(f"\n감지 요약:")
        print(f"- 총 텍스트 박스: {len(detected_texts)}개")
        print(f"- 총 표 영역: {len(table_regions)}개")

# 실행
test_text_detection_visualization()

In [ ]:
# OCR 특이사항
# 1. 0과 O을 잘 구별 못함.
# --> 프롬프트에서 0과 O를 추론하라고 해야할듯

# 2. 영어와 한글이 섞여 존재하면 구별 잘 못함. ex) NH투자증권 -> NH=AAISH

# 3. 표와 표가 아닌 부분의 순서가 이상함 // 해결
# 1) 표가 아닌 부분
# 2) 표인 부분
# 3) 표가 아닌 부분
# 4) 표인 부분
#  >>>> ocr >>>>
# 1) 표가 아닌 부분
# 2) 표가 아닌 부분
# 3) 표인 부분
# 4) 표인 부분

# 4. 표를 읽을때 행을 먼저 읽어야 할 것 같은데 열 순서로 읽는다. // 해결
# --> 프롬프트에서 표 시작과 끝을 보고 표 임을 설명해야할듯

# 변경사항
# 단어 기반 대략 추정 방식으로 수정
# 표 알고리즘 수정
# 표를 인식해서 표 전용 모델을 적용하는 방법 -> 더 성능 안나옴 ㅋ

#### 3. Easy OCR

6분 >> 2분 (GPU: true)

In [ ]:
ocr_params = {
    'lang_list': ['ko', 'en'],    # 인식할 언어 목록
    'gpu': True,                  # GPU 사용 여부

    'text_threshold': 0.7,        # 텍스트로 감지할 신뢰도 임계값 (높을수록 엄격)
    'low_text': 0.4,              # 낮은 신뢰도의 텍스트 필터링 임계값
    'slope_ths': 0.1,             # 기울기 임계값
    'width_ths': 0.6,             # 표 컬럼 분리 개선
    'paragraph': False,            # 표 구조를 유지하기 위해
    'link_threshold': 0.3,        # 텍스트 연결 임계값 완화
}

# easyOCR 리더 초기화
reader = easyocr.Reader(
    lang_list=ocr_params['lang_list'], 
    gpu=ocr_params['gpu']
)

In [ ]:
total_tokens = 0
all_texts = []

# 각 페이지에 대해 OCR 수행
for i, img in enumerate(images):
    print(f"페이지 {i+1} 처리 중...")
    
    # OCR 실행 (파라미터 적용)
    result = reader.readtext(
        np.array(img),
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    # 기본 텍스트 추출
    page_text = ""
    
    for detection in result:
        text = detection[1]
        page_text += text + " "
    
    all_texts.append(page_text)
    
    # 결과 출력
    print(f"페이지 {i+1} 결과:")
    print(page_text)
    
    # 페이지별 단어 수 출력
    page_word_count = len(result)
    total_tokens += page_word_count
    print(f"감지된 텍스트 항목 수: {page_word_count}")
    print("-" * 50)

# 전체 토큰 수 출력
print(f"총 인식된 텍스트 항목 수: {total_tokens}")

# 파일 이름 생성
pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
ocr_engine_name = "easyocr"
txt_file_path = os.path.join(output_folder, f"{pdf_filename}_{ocr_engine_name}_ocr.txt")

# OCR 결과를 텍스트 파일로 저장
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))

print(f"OCR 결과가 {txt_file_path}에 저장되었습니다.")

In [ ]:
def sort_easyocr_results_by_position(result):
    """EasyOCR 결과를 Y좌표(행) 우선, X좌표(열) 순으로 정렬"""
    sorted_results = []
    
    for detection in result:
        if len(detection) == 3:
            bbox, text, confidence = detection
        elif len(detection) == 2:
            bbox, text = detection
            confidence = 1.0
        else:
            continue
            
        # 박스 좌표에서 중심점 계산
        points = np.array(bbox, dtype=np.int32)
        x_center = int(np.mean(points[:, 0]))
        y_center = int(np.mean(points[:, 1]))
        
        sorted_results.append({
            'text': text,
            'confidence': confidence,
            'x_center': x_center,
            'y_center': y_center,
            'bbox': bbox
        })
    
    # Y좌표로 행 그룹핑 (20픽셀 단위)
    rows = {}
    for item in sorted_results:
        row_key = item['y_center'] // 20 * 20
        if row_key not in rows:
            rows[row_key] = []
        rows[row_key].append(item)
    
    # 각 행 내에서 X좌표로 정렬
    final_text = []
    for row_y in sorted(rows.keys()):
        row_items = sorted(rows[row_y], key=lambda x: x['x_center'])
        row_text = ' '.join([item['text'] for item in row_items])
        final_text.append(row_text)
    
    return '\n'.join(final_text)

# 출력 폴더 생성
output_folder = "./easyocr_output"
os.makedirs(output_folder, exist_ok=True)

total_tokens = 0
all_texts = []

# 각 페이지에 대해 OCR 수행
for i, img in enumerate(images):
    print(f"페이지 {i+1} 처리 중...")
    
    # OCR 실행 (파라미터 적용)
    result = reader.readtext(
        np.array(img),
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    # 위치 기반 정렬된 텍스트 추출
    sorted_page_text = sort_easyocr_results_by_position(result)
    all_texts.append(sorted_page_text)
    
    # 결과 출력
    print(f"페이지 {i+1} 정렬된 결과:")
    print(sorted_page_text)
    
    # 페이지별 단어 수 출력
    page_word_count = len(result)
    total_tokens += page_word_count
    print(f"감지된 텍스트 항목 수: {page_word_count}")
    print("-" * 50)

# 전체 토큰 수 출력
print(f"총 인식된 텍스트 항목 수: {total_tokens}")

# 파일 이름 생성
pdf_filename = os.path.splitext(os.path.basename(pdf_path))[0]
ocr_engine_name = "easyocr_sorted"
txt_file_path = os.path.join(output_folder, f"{pdf_filename}_{ocr_engine_name}_ocr.txt")

# OCR 결과를 텍스트 파일로 저장
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write('\n\n===== 페이지 구분선 =====\n\n'.join(all_texts))

print(f"정렬된 OCR 결과가 {txt_file_path}에 저장되었습니다.")

In [ ]:
def enhanced_table_structure_sort_easyocr_results(result, confidence_threshold=0.3):
    """표 구조를 정확히 인식하여 행-열 순서로 정렬하는 고도화된 함수"""
    if not result:
        return ""
    
    # 신뢰도 필터링 및 데이터 준비
    filtered_results = []
    for detection in result:
        if len(detection) == 3:
            bbox, text, confidence = detection
        elif len(detection) == 2:
            bbox, text = detection
            confidence = 1.0
        else:
            continue
        
        if confidence < confidence_threshold:
            continue
            
        # 박스 좌표 정보 추출
        points = np.array(bbox, dtype=np.int32)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)
        x_center = (x_min + x_max) // 2
        y_center = (y_min + y_max) // 2
        
        filtered_results.append({
            'text': text.strip(),
            'confidence': confidence,
            'x_min': x_min,
            'x_max': x_max,
            'y_min': y_min,
            'y_max': y_max,
            'x_center': x_center,
            'y_center': y_center,
            'width': x_max - x_min,
            'height': y_max - y_min
        })
    
    if not filtered_results:
        return ""
    
    print(f"총 {len(filtered_results)}개 텍스트 항목 처리 중...")
    
    # 박스 높이 통계 계산
    heights = [item['height'] for item in filtered_results]
    median_height = np.median(heights)
    
    # 1단계: 표 영역 감지 (X좌표 분포 분석)
    x_centers = [item['x_center'] for item in filtered_results]
    
    # 표의 왼쪽 컬럼 영역 감지 (주로 항목명들이 있는 영역)
    left_boundary = np.percentile(x_centers, 25)  # 25% 지점
    right_boundary = np.percentile(x_centers, 75)  # 75% 지점
    
    print(f"표 영역 추정: 왼쪽={left_boundary:.0f}, 오른쪽={right_boundary:.0f}")
    
    # 2단계: 관대한 Y좌표 기반 행 그룹핑
    row_tolerance = max(25, int(median_height * 0.8))  # 더 관대한 허용 오차
    print(f"행 그룹핑 허용 오차: {row_tolerance}픽셀 (중간 박스 높이: {median_height:.1f})")
    
    # Y좌표 순으로 정렬
    sorted_by_y = sorted(filtered_results, key=lambda x: x['y_center'])
    
    rows = []
    for item in sorted_by_y:
        assigned = False
        
        # 기존 행들과 비교 (더 관대한 조건)
        for row in rows:
            row_avg_y = np.mean([r['y_center'] for r in row])
            y_distance = abs(item['y_center'] - row_avg_y)
            
            # Y축 겹침도 확인
            row_y_min = min(r['y_min'] for r in row)
            row_y_max = max(r['y_max'] for r in row)
            y_overlap = max(0, min(item['y_max'], row_y_max) - max(item['y_min'], row_y_min))
            
            # 더 관대한 조건으로 같은 행 판정
            if y_distance <= row_tolerance or y_overlap > 3:
                row.append(item)
                assigned = True
                break
        
        if not assigned:
            rows.append([item])
    
    print(f"총 {len(rows)}개 행으로 그룹핑됨")
    
    # 3단계: 각 행을 Y좌표로 정렬
    rows.sort(key=lambda row: np.mean([item['y_center'] for item in row]))
    
    # 4단계: 표 구조 분석 및 텍스트 생성
    final_lines = []
    
    for row_idx, row in enumerate(rows):
        # 행 내에서 X좌표로 정렬
        row_items = sorted(row, key=lambda x: x['x_center'])
        
        print(f"행 {row_idx + 1}: {len(row_items)}개 항목")
        for i, item in enumerate(row_items):
            print(f"  {i+1}. '{item['text']}' at ({item['x_center']}, {item['y_center']})")
        
        # 단일 항목 처리
        if len(row_items) == 1:
            final_lines.append(row_items[0]['text'])
            print()
            continue
        
        # 복수 항목 - 표 행 처리
        # 5단계: 지능적인 컬럼 매칭
        left_items = []   # 왼쪽 컬럼 (항목명들)
        right_items = []  # 오른쪽 컬럼 (값들)
        
        for item in row_items:
            if item['x_center'] <= left_boundary + 100:  # 왼쪽 영역 (여유 있게)
                left_items.append(item)
            else:
                right_items.append(item)
        
        # 결과 조합
        row_parts = []
        
        # 왼쪽 항목들 먼저 (항목명)
        if left_items:
            left_text = ' '.join([item['text'] for item in left_items])
            row_parts.append(left_text)
        
        # 구분자 추가
        if left_items and right_items:
            # 간격 계산
            if left_items and right_items:
                gap = min(r['x_center'] for r in right_items) - max(l['x_center'] for l in left_items)
                if gap > 200:
                    row_parts.append('\t|\t')
                elif gap > 100:
                    row_parts.append('\t')
                else:
                    row_parts.append(' ')
        
        # 오른쪽 항목들 (값들)
        if right_items:
            # 오른쪽 항목들 사이의 간격 분석
            for i, item in enumerate(right_items):
                if i > 0:
                    gap = item['x_center'] - right_items[i-1]['x_center']
                    if gap > 150:
                        row_parts.append('\t')
                    elif gap > 50:
                        row_parts.append(' ')
                row_parts.append(item['text'])
        
        final_lines.append(''.join(row_parts))
        print(f"    조합 결과: {''.join(row_parts)}")
        print()
    
    return '\n'.join(final_lines)

# 고도화된 함수로 테스트
print("고도화된 표 구조 인식 EasyOCR 실행...")

# 첫 번째 페이지만 테스트
if images:
    result = reader.readtext(
        np.array(images[1]),
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    sorted_text = enhanced_table_structure_sort_easyocr_results(result)
    
    print("고도화된 표 구조 인식 결과:")
    print("=" * 60)
    print(sorted_text)
    print("=" * 60)

In [ ]:
def visualize_text_sorting_process(image, save_path=None):
    """텍스트 정렬 과정을 상세히 시각화"""
    img_array = np.array(image)
    
    # EasyOCR 결과 가져오기
    result = reader.readtext(
        img_array,
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    # 데이터 준비
    filtered_results = []
    for detection in result:
        if len(detection) >= 2:
            bbox, text = detection[0], detection[1]
            points = np.array(bbox, dtype=np.int32)
            x_center = int(np.mean(points[:, 0]))
            y_center = int(np.mean(points[:, 1]))
            filtered_results.append({
                'text': text.strip(),
                'x_center': x_center,
                'y_center': y_center,
                'bbox': bbox
            })
    
    # 시각화
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(25, 12))
    
    # 원본 감지 순서 (왼쪽)
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title('Original Detection Order (Red Numbers)')
    ax1.axis('off')
    
    for i, item in enumerate(filtered_results):
        points = np.array(item['bbox'], dtype=np.int32)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)
        
        # 박스 그리기
        rect = patches.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, 
                               linewidth=2, edgecolor='red', facecolor='none')
        ax1.add_patch(rect)
        
        # 원본 순서 번호
        ax1.text(x_min, y_min-5, f'{i+1}', fontsize=12, color='red', weight='bold')
    
    # 정렬된 순서 (오른쪽)
    ax2.imshow(img_array, cmap='gray')
    ax2.set_title('Row-Column Sorted Order (Blue Numbers)')
    ax2.axis('off')
    
    # Y좌표로 행 그룹핑
    median_height = np.median([abs(np.array(item['bbox'])[:, 1].max() - np.array(item['bbox'])[:, 1].min()) for item in filtered_results])
    row_tolerance = max(8, int(median_height * 0.4))
    
    rows = []
    sorted_by_y = sorted(filtered_results, key=lambda x: x['y_center'])
    
    for item in sorted_by_y:
        assigned = False
        for row in rows:
            row_avg_y = np.mean([r['y_center'] for r in row])
            if abs(item['y_center'] - row_avg_y) <= row_tolerance:
                row.append(item)
                assigned = True
                break
        if not assigned:
            rows.append([item])
    
    # 행별로 정렬하고 번호 매기기
    rows.sort(key=lambda row: np.mean([item['y_center'] for item in row]))
    
    sorted_order = 1
    colors = ['blue', 'green', 'purple', 'orange', 'brown']
    
    for row_idx, row in enumerate(rows):
        color = colors[row_idx % len(colors)]
        row_items = sorted(row, key=lambda x: x['x_center'])
        
        for item in row_items:
            points = np.array(item['bbox'], dtype=np.int32)
            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            
            # 박스 그리기
            rect = patches.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, 
                                   linewidth=2, edgecolor=color, facecolor='none')
            ax2.add_patch(rect)
            
            # 정렬된 순서 번호
            ax2.text(x_min, y_min-5, f'{sorted_order}', fontsize=12, color=color, weight='bold')
            ax2.text(x_max-15, y_max+15, f'R{row_idx+1}', fontsize=8, color=color, weight='bold')
            sorted_order += 1
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"텍스트 정렬 과정 시각화 저장: {save_path}")
    
    plt.show()

# 정렬 과정 시각화 실행
def test_enhanced_sorting_visualization():
    """향상된 정렬 과정 시각화 테스트"""
    if images:
        print("첫 번째 페이지의 텍스트 정렬 과정을 상세히 시각화합니다...")
        
        viz_folder = "./visualization_output"
        os.makedirs(viz_folder, exist_ok=True)
        
        sorting_save_path = os.path.join(viz_folder, "enhanced_text_sorting_page1.png")
        visualize_text_sorting_process(images[1], sorting_save_path)

# 실행
test_enhanced_sorting_visualization()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_easyocr_detection(image, save_path=None):
    """EasyOCR 텍스트 감지 박스를 시각화"""
    img_array = np.array(image)
    
    # EasyOCR로 텍스트 박스 감지
    result = reader.readtext(
        img_array,
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    # 결과 구조 확인
    print(f"EasyOCR 결과 구조 확인:")
    if result:
        print(f"첫 번째 결과: {result[0]}")
        print(f"결과 길이: {len(result[0]) if result else 0}")
    
    # 시각화 준비
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # 원본 이미지
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    # 텍스트 박스가 있는 이미지
    ax2.imshow(img_array, cmap='gray')
    ax2.set_title(f'EasyOCR Text Detection Boxes ({len(result)} detected)')
    ax2.axis('off')
    
    # 신뢰도별 색상 설정
    def get_color_by_confidence(conf):
        if conf >= 0.8:
            return 'green'      # 높은 신뢰도
        elif conf >= 0.5:
            return 'orange'     # 중간 신뢰도
        else:
            return 'red'        # 낮은 신뢰도
    
    # 텍스트 박스 그리기
    detected_texts = []
    for i, detection in enumerate(result):
        # EasyOCR 결과 구조에 따라 처리
        if len(detection) == 3:
            bbox, text, confidence = detection
        elif len(detection) == 2:
            bbox, text = detection
            confidence = 1.0  # 기본 신뢰도
        else:
            print(f"예상치 못한 결과 구조: {detection}")
            continue
        
        # 박스 좌표 추출 (EasyOCR은 4개 꼭짓점 반환)
        points = np.array(bbox, dtype=np.int32)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)
        
        # 박스 그리기
        color = get_color_by_confidence(confidence)
        rect = patches.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, 
                               linewidth=2, edgecolor=color, facecolor='none')
        ax2.add_patch(rect)
        
        # 신뢰도 텍스트 추가
        ax2.text(x_min, y_min-5, f'{confidence:.2f}', fontsize=8, color=color, weight='bold')
        
        # 텍스트 번호 추가
        ax2.text(x_min+5, y_min+15, f'{i+1}', fontsize=10, color='white', 
                bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.7))
        
        detected_texts.append(f"'{text}' (신뢰도: {confidence:.3f})")
    
    # 범례 추가
    green_patch = patches.Patch(color='green', label='High Confidence (0.8+)')
    orange_patch = patches.Patch(color='orange', label='Medium Confidence (0.5-0.79)')
    red_patch = patches.Patch(color='red', label='Low Confidence (<0.5)')
    ax2.legend(handles=[green_patch, orange_patch, red_patch], loc='upper right')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"EasyOCR 시각화 이미지 저장: {save_path}")
    
    plt.show()
    
    # 감지된 텍스트 목록 출력
    print(f"\nEasyOCR 감지된 텍스트 ({len(detected_texts)}개):")
    for i, text_info in enumerate(detected_texts, 1):
        print(f"{i:2d}. {text_info}")
    
    return detected_texts

def compare_ocr_methods(image, save_path=None):
    """Tesseract와 EasyOCR 감지 결과 비교"""
    img_array = np.array(image)
    
    # Tesseract 결과
    tesseract_data = pytesseract.image_to_data(
        image, 
        lang=OCR_LANG, 
        config=OCR_CONFIG,
        output_type=pytesseract.Output.DICT
    )
    
    # EasyOCR 결과
    easyocr_result = reader.readtext(
        img_array,
        text_threshold=ocr_params['text_threshold'],
        low_text=ocr_params['low_text'],
        slope_ths=ocr_params['slope_ths'],
        width_ths=ocr_params['width_ths'],
        paragraph=ocr_params['paragraph'],
        link_threshold=ocr_params['link_threshold'],
    )
    
    # 시각화 준비
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 10))
    
    # 원본 이미지
    ax1.imshow(img_array, cmap='gray')
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    # Tesseract 결과
    ax2.imshow(img_array, cmap='gray')
    ax2.set_title('Tesseract OCR Detection')
    ax2.axis('off')
    
    tesseract_count = 0
    for i in range(len(tesseract_data['text'])):
        conf = int(tesseract_data['conf'][i])
        text = tesseract_data['text'][i].strip()
        
        if conf > 10 and text:
            x, y, w, h = tesseract_data['left'][i], tesseract_data['top'][i], tesseract_data['width'][i], tesseract_data['height'][i]
            rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='blue', facecolor='none')
            ax2.add_patch(rect)
            tesseract_count += 1
    
    # EasyOCR 결과
    ax3.imshow(img_array, cmap='gray')
    ax3.set_title('EasyOCR Detection')
    ax3.axis('off')
    
    easyocr_count = 0
    for detection in easyocr_result:
        # EasyOCR 결과 구조에 따라 처리
        if len(detection) == 3:
            bbox, text, confidence = detection
        elif len(detection) == 2:
            bbox, text = detection
            confidence = 1.0
        else:
            continue
            
        points = np.array(bbox, dtype=np.int32)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)
        
        rect = patches.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, 
                               linewidth=2, edgecolor='red', facecolor='none')
        ax3.add_patch(rect)
        easyocr_count += 1
    
    plt.suptitle(f'OCR Comparison - Tesseract: {tesseract_count} boxes, EasyOCR: {easyocr_count} boxes', fontsize=16)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"OCR 비교 이미지 저장: {save_path}")
    
    plt.show()
    
    return tesseract_count, easyocr_count

def test_easyocr_visualization():
    """EasyOCR 시각화 테스트"""
    if images:  # 원본 이미지 사용 (전처리 전)
        print("첫 번째 페이지의 EasyOCR 감지 결과를 시각화합니다...")
        
        # 출력 폴더 생성
        viz_folder = "./visualization_output"
        os.makedirs(viz_folder, exist_ok=True)
        
        # EasyOCR 감지 시각화
        easyocr_save_path = os.path.join(viz_folder, "easyocr_detection_page1.png")
        detected_texts = visualize_easyocr_detection(images[0], easyocr_save_path)
        
        print("\n" + "="*70)
        
        # OCR 방법 비교
        compare_save_path = os.path.join(viz_folder, "ocr_comparison_page1.png")
        tesseract_count, easyocr_count = compare_ocr_methods(images[0], compare_save_path)
        
        print(f"\n감지 비교:")
        print(f"- Tesseract 감지 수: {tesseract_count}개")
        print(f"- EasyOCR 감지 수: {easyocr_count}개")

# 실행
test_easyocr_visualization()

#### 토큰 세는법

GPT 토크나이저 로드  
tiktoken 토크나이저 로드